# Employee Fatigue & Productivity Prediction
## Model Training Pipeline using AutoGluon with Hyperparameter Tuning

This notebook builds a complete, production-ready machine learning pipeline using **AutoGluon** to predict employee fatigue levels (`Fatigue_Level`). It performs:
1. Dataset validation and stratified train/test splitting.
2. A baseline training run to automatically select the best model family.
3. Hyperparameter tuning using **Random Search** (sequential trial evaluation) on the selected model family.
4. Model evaluation on the test set.
5. Permutation feature importance analysis.
6. Cleaning and saving only the best-performing model to create a compact, deployment-ready directory.


### Cell 1: Imports
We load the required Python libraries for data handling, model training, evaluation, plotting, and metadata saving.


In [ ]:
import pandas as pd
import numpy as np
import os
import json
import datetime
import shutil
from sklearn.model_selection import train_test_split
from autogluon.tabular import TabularPredictor
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, 
    precision_recall_fscore_support, 
    classification_report, 
    confusion_matrix
)
import warnings
warnings.filterwarnings('ignore')

# Set plotting styles
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11


### Cell 2: Load Dataset
We load the dataset from the project root and print its dimensions.


In [ ]:
csv_path = 'employee_fatigue_productivity_dataset.csv'
df = pd.read_csv(csv_path)

print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()


### Cell 3: Dataset Validation
We verify that the target column `Fatigue_Level` exists, inspect its classes, and display their distribution to check for imbalance.


In [ ]:
target_col = 'Fatigue_Level'

# Assert target exists
assert target_col in df.columns, f"Target column '{target_col}' not found in the dataset!"

print(f"Target column: {target_col}")
print("\nTarget Class Distribution:")
print(df[target_col].value_counts())
print("\nTarget Class Distribution (%):")
print((df[target_col].value_counts(normalize=True) * 100).round(2))


### Cell 4: Train/Test Split
We split the dataset into an 80% training set and a 20% test set. Since the target distribution is imbalanced, we use a stratified split to ensure equal class proportions in both sets.


In [ ]:
train_data, test_data = train_test_split(
    df, 
    test_size=0.20, 
    random_state=42, 
    stratify=df[target_col]
)

print(f"Training set shape: {train_data.shape}")
print(f"Testing set shape: {test_data.shape}")
print("\nTrain target distribution (%):")
print((train_data[target_col].value_counts(normalize=True) * 100).round(2))
print("\nTest target distribution (%):")
print((test_data[target_col].value_counts(normalize=True) * 100).round(2))


### Cell 5: AutoGluon Training & Hyperparameter Tuning
We first run a quick baseline model training pass to identify the best-performing model family. We then delete the baseline directory, clean up the final model directory to ensure proper overwrite, and perform **Hyperparameter Optimization (HPO)** using **Random Search** specifically on the selected best model family.


In [ ]:
model_path = 'models/fatigue_prediction_best_model'
temp_baseline_path = 'models/temp_baseline'

# Clean up existing paths to ensure a clean overwrite if run multiple times
for path in [model_path, temp_baseline_path]:
    if os.path.exists(path):
        print(f"Removing existing directory '{path}' to ensure fresh training...")
        shutil.rmtree(path)

print("\n--- Step 1: Training Baseline Models to Find Best Model Family ---")
# Fit baseline models
baseline_predictor = TabularPredictor(
    label=target_col, 
    path=temp_baseline_path
)

baseline_predictor.fit(
    train_data=train_data, 
    time_limit=180,  # 3 minutes for quick baseline search
    presets='medium_quality'
)

# Identify the best base model (excluding WeightedEnsemble) from baseline leaderboard
leaderboard_baseline = baseline_predictor.leaderboard(silent=True)
base_models = leaderboard_baseline[~leaderboard_baseline['model'].str.contains('Ensemble')]
best_base_model = base_models.iloc[0]['model']
print(f"\nBest baseline base model identified: {best_base_model}")

# Map base model name to AutoGluon's hyperparameter model key
model_key = 'GBM'
if 'LightGBM' in best_base_model or 'GBM' in best_base_model:
    model_key = 'GBM'
elif 'CatBoost' in best_base_model or 'CAT' in best_base_model:
    model_key = 'CAT'
elif 'XGB' in best_base_model:
    model_key = 'XGB'
elif 'RandomForest' in best_base_model or 'RF' in best_base_model:
    model_key = 'RF'
elif 'ExtraTrees' in best_base_model or 'XT' in best_base_model:
    model_key = 'XT'
elif 'NeuralNetFastAI' in best_base_model or 'FASTAI' in best_base_model:
    model_key = 'FASTAI'
elif 'NeuralNetTorch' in best_base_model or 'NN_TORCH' in best_base_model:
    model_key = 'NN_TORCH'

print(f"Selected Model Family for Tuning: {model_key}")

# Clean up the temporary baseline folder
shutil.rmtree(temp_baseline_path)

# Helper function to generate Random Search configurations
def generate_random_hyperparameters(model_family, num_trials=10, seed=42):
    np.random.seed(seed)
    configs = [{}]  # Always include default config as baseline
    for i in range(1, num_trials + 1):
        config = {}
        if model_family == 'GBM':
            config = {
                'learning_rate': float(np.random.choice([0.01, 0.03, 0.05, 0.1, 0.15])),
                'num_leaves': int(np.random.choice([15, 31, 63, 127])),
                'feature_fraction': float(np.random.choice([0.6, 0.7, 0.8, 0.9, 1.0])),
                'min_data_in_leaf': int(np.random.choice([3, 10, 20, 50])),
                'ag_args': {'name_suffix': f'_RandomTrial{i}'}
            }
        elif model_family == 'CAT':
            config = {
                'learning_rate': float(np.random.choice([0.01, 0.03, 0.05, 0.1])),
                'depth': int(np.random.choice([4, 6, 8, 10])),
                'l2_leaf_reg': float(np.random.choice([1, 3, 5, 10])),
                'ag_args': {'name_suffix': f'_RandomTrial{i}'}
            }
        elif model_family == 'XGB':
            config = {
                'learning_rate': float(np.random.choice([0.01, 0.03, 0.05, 0.1, 0.15])),
                'max_depth': int(np.random.choice([4, 6, 8, 10])),
                'subsample': float(np.random.choice([0.6, 0.7, 0.8, 0.9, 1.0])),
                'colsample_bytree': float(np.random.choice([0.6, 0.7, 0.8, 0.9, 1.0])),
                'ag_args': {'name_suffix': f'_RandomTrial{i}'}
            }
        elif model_family == 'RF':
            config = {
                'max_depth': int(np.random.choice([10, 15, 20, 25, 30])),
                'min_samples_leaf': int(np.random.choice([1, 2, 4, 8])),
                'ag_args': {'name_suffix': f'_RandomTrial{i}'}
            }
        elif model_family == 'XT':
            config = {
                'max_depth': int(np.random.choice([10, 15, 20, 25, 30])),
                'min_samples_leaf': int(np.random.choice([1, 2, 4, 8])),
                'ag_args': {'name_suffix': f'_RandomTrial{i}'}
            }
        elif model_family == 'FASTAI':
            layers_options = [[200, 100], [100, 50], [500, 200, 100]]
            chosen_idx = np.random.choice(len(layers_options), p=[0.4, 0.4, 0.2])
            config = {
                'layers': layers_options[chosen_idx],
                'emb_drop': float(np.random.choice([0.0, 0.1, 0.2, 0.3])),
                'ag_args': {'name_suffix': f'_RandomTrial{i}'}
            }
        elif model_family == 'NN_TORCH':
            config = {
                'learning_rate': float(np.random.choice([1e-4, 1e-3, 5e-3, 1e-2])),
                'weight_decay': float(np.random.choice([1e-6, 1e-4, 1e-2])),
                'ag_args': {'name_suffix': f'_RandomTrial{i}'}
            }
        else:
            config = {
                'ag_args': {'name_suffix': f'_RandomTrial{i}'}
            }
        configs.append(config)
    return configs

# Generate hyperparameter configs for Random Search HPO
random_configs = generate_random_hyperparameters(model_key, num_trials=10, seed=42)

print(f"\n--- Step 2: Running Sequential Random Search HPO on {model_key} ---")
print(f"Generated {len(random_configs)} random configurations to evaluate.")

# Initialize the final predictor
predictor = TabularPredictor(
    label=target_col, 
    path=model_path
)

# Fit models sequentially using our generated hyperparameter list (avoiding Ray JobObject issues on Windows)
predictor.fit(
    train_data=train_data, 
    hyperparameters={model_key: random_configs}
)


### Cell 6: Leaderboard
We generate the model comparison leaderboard on the test set to view and rank the tuned configurations.


In [ ]:
leaderboard = predictor.leaderboard(test_data, silent=False)
leaderboard


### Cell 7: Best Model Information
We display the name of the best-tuned model, its validation score, and training time.


In [ ]:
best_model_name = predictor.model_best

# Extract validation score from the leaderboard
best_model_info = leaderboard[leaderboard['model'] == best_model_name].iloc[0]
val_score = best_model_info['score_val']
fit_time = best_model_info['fit_time']

print(f"Best Model Selected: {best_model_name}")
print(f"Best Model Validation Score: {val_score:.4f}")
print(f"Best Model Fit Time: {fit_time:.2f} seconds")


### Cell 8: Test Evaluation
We evaluate the predictor on the test set and calculate key classification metrics: Accuracy, Precision, Recall, and F1 Score (Macro-averaged).


In [ ]:
y_test = test_data[target_col]
X_test = test_data.drop(columns=[target_col])

# Make predictions
y_pred = predictor.predict(X_test)

# Calculate metrics
acc = accuracy_score(y_test, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='macro')

print("Test Evaluation Metrics:")
print(f"Accuracy:        {acc:.4f}")
print(f"Precision (Macro): {precision:.4f}")
print(f"Recall (Macro):    {recall:.4f}")
print(f"F1 Score (Macro):  {f1:.4f}")


### Cell 9: Classification Report
We display the class-wise classification report to observe precision, recall, and F1 score for each class (`Low`, `Medium`, `High`).


In [ ]:
print("Classification Report:")
print(classification_report(y_test, y_pred))


### Cell 10: Confusion Matrix
We plot a normalized confusion matrix using Seaborn to visualize prediction errors and class confusions.


In [ ]:
labels = ['Low', 'Medium', 'High']
cm = confusion_matrix(y_test, y_pred, labels=labels)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    cmap='Blues', 
    xticklabels=labels, 
    yticklabels=labels
)
plt.title('Confusion Matrix - Fatigue Level Prediction')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.show()


### Cell 11: Feature Importance Table
We compute permutation feature importance using the test set to understand which features contribute most to the model's accuracy.


In [ ]:
feature_importance = predictor.feature_importance(test_data)
print("Top 20 Feature Importances:")
feature_importance.head(20)


### Cell 12: Feature Importance Plot
We visualize the top 20 features in a bar chart.


In [ ]:
top_20_features = feature_importance.head(20)
plt.figure(figsize=(12, 8))
sns.barplot(
    data=top_20_features, 
    x='importance', 
    y=top_20_features.index, 
    palette='viridis'
)
plt.title('Top 20 Features Importance (AutoGluon Best Model)')
plt.xlabel('Importance (Score Drop on Permutation)')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()


### Cell 13: Save Best Model Only
To create a compact deployment-ready model folder, we delete all models that are not the best model (or its dependencies) and call `save_space()` to clean up auxiliary and training data files.


In [ ]:
print(f"Cleaning up models in: {model_path}")
# Keep only the best model and its dependencies, delete others from disk
predictor.delete_models(models_to_keep='best', dry_run=False)

# Remove training data and intermediate artifacts to reduce space
predictor.save_space()
print("Artifact cleaning completed! Compact deployment directory created.")


### Cell 14: Save Metadata
We create and save `model_metadata.json` inside the best model directory containing details about the training process and the model's configuration.


In [ ]:
metadata = {
    "target_column": target_col,
    "problem_type": predictor.problem_type,
    "best_model_name": best_model_name,
    "validation_score": float(val_score),
    "training_date": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "feature_count": len(predictor.features()),
    "autogluon_version": predictor.info().get('version', 'unknown')
}

metadata_path = os.path.join(model_path, 'model_metadata.json')
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=4)

print(f"Model metadata saved to {metadata_path}")
# Display saved metadata
print(json.dumps(metadata, indent=4))


### Cell 15: Final Training Summary
We output a clean training summary detailing the dataset shapes, model selection, scores, and save locations.


In [ ]:
print("=" * 60)
print("FINAL TRAINING SUMMARY")
print("=" * 60)
print(f"Dataset Shape:         {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Training Rows:         {train_data.shape[0]}")
print(f"Testing Rows:          {test_data.shape[0]}")
print(f"Best Model Selected:   {best_model_name}")
print(f"Validation Score:      {val_score:.4f}")
print(f"Test Score (Accuracy): {acc:.4f}")
print(f"Number of Features:    {len(predictor.features())}")
print(f"Model Save Location:   {model_path}")
print("=" * 60)
